# Amazon ML Challenge 2026 — Business Entity Resolution
## Experiment Tracking Notebook

Tracks pipeline setup, blocking/candidate-generation metrics, model training/validation
metrics, and leaderboard submissions across iterations, so future rounds can compare
against what's already been tried instead of re-deriving it from scratch.

**Current public leaderboard score: 0.802** (submission after the blocking/normalization
fix round described below).


## 1. Pipeline Architecture

4-stage cascaded hybrid entity-resolution pipeline:

1. **Disk-backed SQLite blocking index** — multi-pass, order-invariant blocking keys
   (`tok_` name tokens, `num_` house/street number + address token, `zip_` postal-code-shaped
   numbers alone, `bg_` unordered address-token bigram), partitioned strictly by country.
   Built once over the full S2/S3 candidate pool, capped per-key posting-list length
   (`MAX_TOKEN_POSTINGS`) to drop near-stopword keys.
2. **Stage 2 — LightGBM screener**: gradient-boosted trees over 18 dense tabular
   name/address/number similarity features.
3. **Stage 3 — TensorFlow/Keras deep residual specialist** (ResNet-MLP, Swish + BatchNorm +
   Dropout), trained on the same data plus actively-mined hard negatives (LightGBM false
   positives with $P \ge 0.35$).
4. **Stage 4 — Global duplicate resolution**: a candidate accepted by more than one S1
   entity is kept only under whichever one scored it highest (ground truth never assigns
   one candidate to two S1 entities).

Ensemble: $P_{final} = 0.70 \times P_{LGBM} + 0.30 \times P_{TF}$ in the ambiguous zone
($0.20 \le P_{LGBM} < 0.88$), decided at $\tau = 0.70$.

**Environment constraints:** 7.4 GB RAM (WSL2 default cap on a 16 GB host), 8 GB VRAM.
The blocking index is deliberately kept small enough to fit in page cache at this scale —
see the trade-off note in Section 2.


In [1]:
import pandas as pd

pd.set_option("display.max_colwidth", None)


## 2. Candidate Generation (Blocking) Metrics

Measured on a 2,000-entity validation sample.


In [2]:
blocking_metrics = pd.DataFrame([
    {"stage": "Before this round's fixes", "avg_candidates_per_s1": None,
     "pair_recall": 0.675, "oracle_f0.5_ceiling": 0.82, "index_size_gb": "~5.5 (train)"},
    {"stage": "Richer key design (every num x top-2 addr tok, 4-token bigram combos)",
     "avg_candidates_per_s1": None, "pair_recall": 0.864, "oracle_f0.5_ceiling": 0.939,
     "index_size_gb": "8.22 (train) / 9.4+ (test) - exceeded 7.4GB RAM cap, caused thrashing"},
    {"stage": "Shipped (size-safe trim: num x 1 addr tok, bg = 1 sorted pair)",
     "avg_candidates_per_s1": 25.5, "pair_recall": 0.771, "oracle_f0.5_ceiling": 0.888,
     "index_size_gb": "4.84 (test, post-VACUUM)"},
])
blocking_metrics


,stage,avg_candidates_per_s1,pair_recall,oracle_f0.5_ceiling,index_size_gb
0,Before this round's fixes,NaN,0.675,0.820,~5.5 (train)
1,"Richer key design (every num x top-2 addr tok, 4-token bigram combos)",NaN,0.864,0.939,"8.22 (train) / 9.4+ (test) - exceeded 7.4GB RAM cap, caused thrashing"
2,"Shipped (size-safe trim: num x 1 addr tok, bg = 1 sorted pair)",25.5,0.771,0.888,"4.84 (test, post-VACUUM)"


**Trade-off note:** the richer key design measured higher recall/ceiling but grew the
on-disk index past available RAM, causing page-cache thrashing (31-34 ent/s live, a
~14-15hr projected full run vs. the ~200-300 ent/s baseline). The shipped design trades
some of that recall for an index that reliably fits in memory. See `Documentation_template.md`
Section 5 for the full bug list and root causes.


## 3. Model Training & Validation Metrics

On an 8,000-entity disjoint held-out validation sample (S1 entities never seen by
either model during training).


In [3]:
results = pd.DataFrame([
    {"configuration": "Before this round of fixes (LGBM only, 15k-entity train sample)",
     "best_threshold": 0.75, "macro_f0.5": 0.7547, "singleton_acc": 0.8619},
    {"configuration": "LightGBM screener alone (post-fix, 80k-entity train sample)",
     "best_threshold": 0.70, "macro_f0.5": 0.8172, "singleton_acc": 0.8843},
    {"configuration": "TensorFlow specialist alone (post-fix)",
     "best_threshold": 0.55, "macro_f0.5": 0.8112, "singleton_acc": 0.8802},
    {"configuration": "Full ensemble + Stage 4 dedup (shipped configuration)",
     "best_threshold": 0.70, "macro_f0.5": 0.8177, "singleton_acc": 0.9153},
])
results


,configuration,best_threshold,macro_f0.5,singleton_acc
0,"Before this round of fixes (LGBM only, 15k-entity train sample)",0.75,0.7547,0.8619
1,"LightGBM screener alone (post-fix, 80k-entity train sample)",0.70,0.8172,0.8843
2,TensorFlow specialist alone (post-fix),0.55,0.8112,0.8802
3,Full ensemble + Stage 4 dedup (shipped configuration),0.70,0.8177,0.9153


**Training sample sizes:** 80,000 S1 entities for both the LightGBM screener and the
TensorFlow specialist (up from 15,000/2,544 hard negatives before this round).
Hard negatives for the specialist: ~22,000 actively mined from LightGBM false positives
($P \ge 0.35$) against the fixed blocking output (up from 2,544).

**Features (18 dense tabular):** Levenshtein ratio, partial ratio, token sort ratio, token
set ratio, Jaro-Winkler, first-token-exact flag, length discrepancy (name side); address
token set ratio, address Jaccard, address partial ratio, address length diff (address
side); exact number set match, number Jaccard overlap, number overlap count, number
conflict flag (numeric disambiguation); source (S2 vs S3) indicator.

**Top LightGBM feature by gain:** `addr_token_set`, followed by `name_partial_ratio` and
`name_ratio`.


## 4. Full Test-Set Inference Run

Real 1,732,544-entity test set (public test data), using the shipped size-safe blocking
index (4.84 GB, post-`VACUUM`).


In [4]:
inference_run = {
    "total_s1_entities": 1732544,
    "wall_clock_minutes": 113.07,
    "steady_state_throughput_ent_per_s": 255,
    "raw_matches_found": 4333728,
    "matches_after_stage4_dedup": 4242555,
    "duplicate_claimed_matches_removed": 91173,
    "duplicate_removed_pct": 2.10,
    "validator_result": "PASS - no blocking issues found",
}
inference_run


{'total_s1_entities': 1732544,
 'wall_clock_minutes': 113.07,
 'steady_state_throughput_ent_per_s': 255,
 'raw_matches_found': 4333728,
 'matches_after_stage4_dedup': 4242555,
 'duplicate_claimed_matches_removed': 91173,
 'duplicate_removed_pct': 2.1,
 'validator_result': 'PASS - no blocking issues found'}

## 5. Leaderboard Submissions

| Date | Threshold | Config | Public Score | Notes |
|---|---|---|---|---|
| (initial) | 0.88 | LGBM-only, 15k train sample, buggy non-deterministic blocking keys | **0.65** | Root cause: see bug list in `Documentation_template.md` Sec. 5 |
| 2026-09-26 | 0.70 | Full ensemble (LGBM+TF) + Stage 4 dedup, size-safe blocking, 80k train sample | **0.802** | Current best. Validation Macro F0.5 was 0.8177; public score landed close to that, confirming the validation split generalizes reasonably well despite France being unseen at training time. |

**Gap to close next:** validation ceiling with a perfect matcher is 0.888 (blocking-limited);
current ensemble is 0.8177 on validation / 0.802 public. Two independent levers, not
mutually exclusive:
- **Blocking recall** (currently 77.1% pair recall) — fuzzy/embedding-based retrieval to
  recover some of the 86.4%-recall design's gain without the memory blowup (e.g. more RAM
  via `.wslconfig`, or a smarter key design that doesn't scale combinatorially).
- **Matching model** — closing the ~7pt gap between the ensemble (0.8177) and the recall
  ceiling (0.888) on the *same* candidates, via richer features or a better-tuned specialist.


## 6. Bugs Fixed This Round (in rough order of impact)

1. **Non-deterministic blocking keys** — `list(nums)[0]` on a Python `set` depended on
   per-process iteration order; the same address could index/query under different keys
   across runs. Fixed with `sorted(extract_numbers(...))`.
2. **Order-dependent address keys** — only used the *first* street number and *first two*
   (order-dependent) address tokens; reordered/multi-number addresses were frequently
   missed. Fixed with an order-invariant key generator shared identically between
   index-build and query time.
3. **Untruncated/unfiltered posting lists** — common keys silently dropped true matches via
   an unordered `LIMIT` instead of being filtered out up front (`MAX_TOKEN_POSTINGS` cap).
4. **Stale decision threshold** — deployed at 0.88, model's own validation sweep said
   0.70-0.75.
5. **Missing French normalization** (~15% of test set, unseen at training) — un-folded
   accents (`Àmicale` != `Amicale`) and un-stripped dotted acronyms (`S.A.S.` -> `s a s`
   not recognized as `sas`).
6. **Undersized training sample** — only 15,000 of 2.2M available S1 entities used; now
   80,000 for both screener and specialist.
7. **No global duplicate resolution** — a candidate could be accepted by >1 S1 entity even
   though ground truth never does this; Stage 4 now resolves via highest-score-wins.
8. **Redundant re-normalization** — `extract_pair_features()` re-normalized both sides'
   name/address from raw text on *every* candidate pair (~25x redundant per S1 entity);
   now reuses already-computed normalized strings persisted at index-build time.
9. **(Performance) Un-batched key lookups** — one SQL round-trip per key (~33ms/record,
   15+ hours projected) fixed by batching into one `key IN (...)` query per record.
10. **(Performance) Index-size blowup from the recall fix** — richer keys grew the index
    from ~5.5GB to 9.4GB, past the 7.4GB RAM cap, causing page-cache thrashing (31-34 ent/s
    vs. ~200-300 ent/s baseline). Fixed by trimming the two key types responsible for 90%
    of row growth (`bg_`/`num_`), trading recall (86.4% -> 77.1%) for a reliably-in-RAM index.
11. **(Performance) Catastrophic `NOT IN` subquery** — the key-frequency-capping step's
    first fix attempt (`WHERE (country,key) NOT IN (SELECT ... GROUP BY ...)`) hung 90+
    minutes; SQLite re-executes the aggregated subquery per outer row instead of
    materializing it once. Fixed with materialize-into-indexed-temp-table +
    `LEFT JOIN ... WHERE ... IS NULL` anti-join (199.7s at full scale).
12. **Guardrails added** — `build_index()` logs row counts/DB size and warns if it exceeds
    a safe fraction of detected RAM; `pipeline.py` runs a retrieval throughput smoke-test
    (steady-state, second-half-of-window measurement) before committing to the full
    streaming loop, aborting with a clear error if it's far below the expected range.

Full detail and measurements: `Documentation_template.md`, Section 5.


## 7. Open Questions / Next Ideas

- Try raising the WSL2 RAM cap (`.wslconfig`, `memory=12GB` + `wsl --shutdown`) to afford
  a richer blocking-key design (recover toward the 86.4%/0.939-ceiling version) without
  page-cache thrashing.
- Investigate whether `THRESHOLD_AUTO_ACCEPT`/`THRESHOLD_AUTO_REJECT` in `config.py`
  (0.88/0.15) are still meaningfully connected to `pipeline.py`'s routing logic, which
  currently hardcodes 0.20/0.88 for the ensemble's ambiguous zone — reconcile or remove
  the unused config values.
- Consider fuzzy/embedding-based candidate retrieval as a second blocking pass to close
  some of the remaining ~23% recall gap without a combinatorial key blowup.
- Re-run the 2,000-entity oracle-recall check and full validation sweep after any future
  blocking/feature change, since threshold and ensemble weights are sensitive to both.
